In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")

In [3]:
import torch
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.float
device   = 'cuda'
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

In [4]:
import numpy as np

In [5]:
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN

tokenizer = initialize_tokenizer(model_id)

In [6]:
blocksworld_type = "4-blocks"

dataset = load_dataset(f"dmitriihook/deepseek-r1-qwen-32b-planning-{blocksworld_type}")["train"]

In [7]:
import json

with open("blocksworld-4-self-probing-parsed-big-v2.json", "r") as f:
    labels_dataset = json.load(f)

# labels_dataset = load_dataset(f"dmitriihook/blocksworld-6-self-probing-parsed")["train"]

In [8]:
from collections import defaultdict
from typing import Optional

def stacks_to_pairs(stacks: list[list[str]]) -> tuple[dict, dict, Optional[str]]:
    above = {}
    below = {}

    for stack in stacks:
        for i, block in enumerate(stack):
            if i == 0:
                above[block] = "sky"
            else:
                above[block] = stack[i - 1]
                below[stack[i - 1]] = block
            below[block] = "table"
        
    return above, below

def check_if_block(block: str) -> bool:
    return block in ["A", "B", "C", "D"]

def check_stacks(stacks: list[list[str]], n_blocks: int) -> bool:
    blocks = set()
    for stack in stacks:
        for block in stack:
            if not check_if_block(block):
                return False
            blocks.add(block)
    return len(blocks) == n_blocks

labels_dict = defaultdict(dict)
for item in labels_dataset:
    idx = item["idx"]
    line_n = item["line_n"]
    parsed = item["parsed"]

    if parsed is None:
        continue

    if "blocks" not in parsed:
        continue
    
    if not check_stacks(parsed["blocks"], 4):
        continue

    labels_dict[idx][line_n] = item


In [9]:
import re
from collections import defaultdict

def parse_blocks(text):
    initial_state = []
    goal_state = []
    
    # Extract the initial conditions and goal state
    initial_match = re.search(r'As initial conditions I have that:(.*?)My goal is for the following to be true:', text, re.DOTALL)
    goal_match = re.search(r'My goal is for the following to be true:(.*?)\n\n', text, re.DOTALL)

    if initial_match:
        initial_conditions = re.findall(r'Block [A-Z] is on top of Block [A-Z]', initial_match.group(1))
        init_table_blocks = re.findall(r'Block ([A-Z]) is on the table', initial_match.group(1))
        initial_state = process_conditions(initial_conditions)

    
    if goal_match:
        goal_conditions = re.findall(r'Block [A-Z] is on top of Block [A-Z]', goal_match.group(1))
        goal_table_blocks = re.findall(r'Block ([A-Z]) is on the table', goal_match.group(1))
        goal_state = process_conditions(goal_conditions)

    
    return (initial_state, init_table_blocks), (goal_state, goal_table_blocks)

def process_conditions(conditions):
    pairs = {}
    
    for cond in conditions:
        block, below = re.findall(r'Block ([A-Z])', cond)
        pairs[block] = below
    
    return pairs


item = dataset[2]["query"]
stmt = item.split("[STATEMENT]")[-1].strip()

initial_state, goal_state = parse_blocks(stmt)
initial_state, goal_state

(({'B': 'C', 'C': 'D'}, ['A', 'D']), ({'A': 'C', 'C': 'D', 'D': 'B'}, []))

In [10]:
def collect_all_blocks(initial_state):
    all_blocks = list(initial_state[0].keys())
    all_blocks.extend(initial_state[1])
    all_blocks.extend(initial_state[0].values())
    return list(set(all_blocks))

def state_to_pairs(state, all_blocks):
    pairs, _ = state
    below = {}

    for block, below_block in pairs.items():
        below[block] = below_block

    for block in all_blocks:
        if block not in below:
            below[block] = "table"

    above = {}

    for block, below_block in below.items():
        if below_block != "table":
            above[below_block] = block

    for block in all_blocks:
        if block not in above:
            above[block] = "sky"
    
    return above, below

In [11]:
def state_compare(above1, below1, above2, below2):
    for block in above1:
        if above1[block] != above2[block]:
            return False
    for block in below1:
        if below1[block] != below2[block]:
            return False
        
    return True

In [12]:
from pathlib import Path

def load_dataset_from_file(domain_name, task_name):
    prompt_dir = Path(f"./cot-planning/results/{domain_name}/deepseek-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)


task_name = "plan_generation_po"
domain_name = "blocksworld_4_blocks"
eval_results = load_dataset_from_file(domain_name, task_name)["instances"]

In [13]:
eval_results = {x["dataset_idx"]: x for x in eval_results}

In [14]:
eval_results[0]

{'instance_id': '4_1',
 'query': 'I am playing with a set of blocks where I need to arrange the blocks into stacks. Here are the actions I can do\n\nPick up a block\nUnstack a block from on top of another block\nPut down a block\nStack a block on top of another block\n\nI have the following restrictions on my actions:\nI can only pick up or unstack one block at a time.\nI can only pick up or unstack a block if my hand is empty.\nI can only pick up a block if the block is on the table and the block is clear. A block is clear if the block has no other blocks on top of it and if the block is not picked up.\nI can only unstack a block from on top of another block if the block I am unstacking was really on top of the other block.\nI can only unstack a block from on top of another block if the block I am unstacking is clear.\nOnce I pick up or unstack a block, I am holding the block.\nI can only put down a block that I am holding.\nI can only stack a block on top of another block if I am hol

In [15]:
n_blocks = 4

In [16]:
n_rows = 1500

In [ ]:
data_to_process = []

take_prob = 0.5

# batch_size = 100

for idx, row in enumerate(tqdm(dataset.select(range(n_rows)))):
    if not eval_results[idx]["llm_correct"]:
        continue
    query = row["query"]
    stmt = query.split("[STATEMENT]")[-1].strip()
    initial_state, goal_state = parse_blocks(stmt)
    all_blocks = collect_all_blocks(initial_state)
    i_above, i_below = state_to_pairs(initial_state, all_blocks)
    g_above, g_below = state_to_pairs(goal_state, all_blocks)

    generation = row["generation"]

    text = ""

    # prompts = []

    for line_n, line in enumerate(generation.split("\n\n")):
        _text = text
        text = text + line + "\n\n"
        if line_n < 10 or len(line) < 30:
            continue
        # if line_n >= 20 and len(line) >= 50:
        #     continue

        if line_n not in labels_dict[idx]:
            continue

        above, below = stacks_to_pairs(labels_dict[idx][line_n]["parsed"]["blocks"])
        if state_compare(i_above, i_below, above, below):
            if np.random.rand() > take_prob:
                continue
        if state_compare(g_above, g_below, above, below):
            if np.random.rand() > take_prob:
                continue

        
        # _text = text + "Now, the stacks are:\n\n-"
        
        _query = row["distilabel_metadata"]["raw_input_text_generation_0"][0]

        messages = [
            _query,
            {"role": "assistant", "content": _text}
        ]
        tokens_pre = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)[:-1]
        
        messages = [
            _query,
            {"role": "assistant", "content": text}
        ]
        
        tokens_post = tokenizer.apply_chat_template(messages, tokenize=True, add_special_tokens=False)[:-1]

        data_to_process.append({
            "idx": idx,
            "line_n": line_n,
            "tokens_pre": tokens_pre,
            "tokens_post": tokens_post,
            "above": above,
            "below": below
        })

  0%|          | 0/1500 [00:00<?, ?it/s]

In [ ]:
len(data_to_process)

29911

In [ ]:
from vllm import LLM

llm = LLM(model=model_id, task="reward", tensor_parallel_size=8)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

INFO 02-26 18:20:58 __init__.py:190] Automatically detected platform cuda.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

INFO 02-26 18:21:10 config.py:1401] Defaulting to use mp for distributed inference
WARNING 02-26 18:21:10 arg_utils.py:1145] The model has a long context length (131072). This may cause OOM errors during the initial memory profiling phase, or result in low performance due to small KV cache space. Consider setting --max-model-len to a smaller value.
INFO 02-26 18:21:10 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(gu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


(VllmWorkerProcess pid=1048367) INFO 02-26 18:21:11 multiproc_worker_utils.py:229] Worker ready; awaiting tasks
INFO 02-26 18:21:20 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1048355) INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1048337) INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1048340) INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1048367) INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1048350) (VllmWorkerProcess pid=1048360) INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=1048345) INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
INFO 02-26 18:21:21 cuda.py:230] Using Flash Attention backend.
INFO 02-26 18:21:26 utils.py:950] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=1048355) (VllmWorkerProcess pid=1048345) INFO 02-26 18:21:

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]


(VllmWorkerProcess pid=1048350) INFO 02-26 18:21:39 model_runner.py:1115] Loading model weights took 7.5269 GB
INFO 02-26 18:21:40 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=1048367) INFO 02-26 18:21:40 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=1048355) INFO 02-26 18:21:40 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=1048345) INFO 02-26 18:21:41 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=1048337) INFO 02-26 18:21:41 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=1048340) INFO 02-26 18:21:41 model_runner.py:1115] Loading model weights took 7.5269 GB
(VllmWorkerProcess pid=1048360) INFO 02-26 18:21:42 model_runner.py:1115] Loading model weights took 7.5269 GB


(VllmWorkerProcess pid=1048345) (VllmWorkerProcess pid=1048337) (VllmWorkerProcess pid=1048367) (VllmWorkerProcess pid=1048355) (VllmWorkerProcess pid=1048350) INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting
INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting
INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting
INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting
INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting
(VllmWorkerProcess pid=1048340) (VllmWorkerProcess pid=1048360) INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting
INFO 02-26 18:29:21 multiproc_worker_utils.py:253] Worker exiting


In [ ]:
from vllm import TokensPrompt

batch_size = 200

last_hidden_states = []

for i in tqdm(range(0, n_rows, batch_size)):
    batch = dataset.select(range(i, min(i + batch_size, n_rows)))
    tokens = [tokenize_blocksworld_generation(tokenizer, row)[0] for row in batch]

    tokens = [TokensPrompt(prompt_token_ids=t) for t in tokens]

    output = llm.encode(tokens)

    for x in output:
        hs = x.outputs.data
        last_hidden_states.append(hs.cpu().to(torch.float16).numpy())

  0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 100/100 [00:09<00:00, 10.34it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [ ]:
block_positions = []

all_blocks = [chr(65 + i) for i in range(n_blocks)]

for row in dataset.select(range(n_rows)):
    tokens = tokenize_blocksworld_generation(tokenizer, row)[0]
    row_block_positions = {}

    for block in all_blocks:
        block_token = tokenizer.encode(" " + block, add_special_tokens=False)[0]
        _block_positions = np.where(tokens == block_token)[0]
        row_block_positions[block] = _block_positions

    block_positions.append(row_block_positions)

In [ ]:
from torch.utils.data import Dataset

act2int = {
    "put down": 0,
    "pick up": 1,
    "stack": 2,
    "unstack": 3
}

def block2int(block):
    if block == "table":
        return n_blocks
    if block == "sky":
        return n_blocks + 1
    
    return ord(block) - ord("A")

def int2block(i):
    if i == n_blocks:
        return "table"
    if i == n_blocks + 1:
        return "sky"
    
    return chr(i + ord("A"))

n_prev_tokens = 50

def state_to_label(state, top_block, bottom_block):
    above, below, hand = state
    return int(below[top_block] == bottom_block)
    top_block_stack = [top_block]
    while top_block_stack[-1] != "table":
        top_block_stack.append(below[top_block_stack[-1]])
    top_block_stack = top_block_stack[1:]

    return int(bottom_block in top_block_stack)

negative_dropout = 0.5

def collect_block_positions(items, top_blocks, bottom_blocks):
    new_items = []

    for top_block in top_blocks:
        for bottom_block in bottom_blocks:
            if top_block == bottom_block:
                continue
            for item in items:
                prev_pos = len(item["tokens_pre"])
                post_pos = len(item["tokens_post"])

                label = state_to_label((item["above"], item["below"], None), top_block, bottom_block)

                if label == 0:
                    if np.random.rand() > negative_dropout:
                        continue

                _block_positions = block_positions[item["idx"]]

                top_block_pos = _block_positions[top_block]
                bottom_block_pos = _block_positions[bottom_block]

                top_block_pos = top_block_pos[top_block_pos < post_pos]
                bottom_block_pos = bottom_block_pos[bottom_block_pos < post_pos]

                top_block_pos = top_block_pos[top_block_pos > prev_pos]
                bottom_block_pos = bottom_block_pos[bottom_block_pos > prev_pos]

                if len(top_block_pos) == 0 or len(bottom_block_pos) == 0:
                    continue

                new_items.append({
                    "idx": item["idx"],
                    "line_n": item["line_n"],
                    "top_positions": top_block_pos,
                    "bottom_positions": bottom_block_pos,
                    "top_block": top_block,
                    "bottom_block": bottom_block,
                    "above": item["above"],
                    "below": item["below"],
                    "label": label
                })

    return new_items

class StepProbeDataset(Dataset):
    def __init__(self, items, n_layer, top_blocks, bottom_blocks):
        self.items = collect_block_positions(items, top_blocks, bottom_blocks)
        self.hidden_states = last_hidden_states
        self.n_layer = n_layer
        self.top_blocks = top_blocks
        self.bottom_blocks = bottom_blocks

    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        item = self.items[idx]

        top_positions = item["top_positions"]
        bottom_positions = item["bottom_positions"]

        top_representations = self.hidden_states[item["idx"]][top_positions].mean(axis=0)
        bottom_representations = self.hidden_states[item["idx"]][bottom_positions].mean(axis=0)

        # hidden_states = self.hidden_states[item["idx"]][post_pos-10:post_pos + 1]
        hidden_states = top_representations - bottom_representations
        above, below = item["above"], item["below"]
        # print(above, below)
        return {
            "input": hidden_states,
            "labels": item["label"]
        }


In [23]:
training_data = data_to_process

In [33]:
train_test_split = 0.9    
n_train = int(len(training_data) * train_test_split)

train_items = training_data[:n_train]
test_items = training_data[n_train:]

train_dataset = StepProbeDataset(train_items, 0, top_blocks=["D"], bottom_blocks=["A"])
test_dataset = StepProbeDataset(test_items, 0, top_blocks=["D"], bottom_blocks=["A"])

In [34]:
train_dataset.items[4]

{'idx': 0,
 'line_n': 36,
 'top_positions': array([1945, 1955]),
 'bottom_positions': array([1947, 1957]),
 'top_block': 'D',
 'bottom_block': 'A',
 'above': {'D': 'sky', 'A': 'D', 'B': 'A', 'C': 'sky'},
 'below': {'D': 'A', 'A': 'B', 'B': 'table', 'C': 'table'},
 'label': 1}

In [35]:
class StepProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        # self.fc = torch.nn.Linear(input_size, hidden_size)
        # self.fc2 = torch.nn.Linear(hidden_size, n_blocks * (n_blocks + 2) * 2)
        # self.fc2 = torch.nn.Linear(input_size, n_blocks * (n_blocks + 2) * 2)
        self.fc2 = torch.nn.Linear(input_size, 2)
        # self.dropout = torch.nn.Dropout(0.1)
        
    def forward(self, x):
        # x = self.fc(x)
        # x = torch.nn.functional.relu(x)
        # x = self.dropout(x)
        x = self.fc2(x)
        return x
        # return x.view(-1, n_blocks + 2, n_blocks * 2)

In [36]:
class AHProbe(torch.nn.Module):
    def __init__(self, input_size, hidden_size, n_blocks):
        super().__init__()
        self.fc = torch.nn.Linear(input_size, hidden_size)
        self.fc2 = torch.nn.Linear(hidden_size, n_blocks * (n_blocks + 2) * 2)
        # self.fc2 = torch.nn.Linear(hidden_size, n_blocks + 2)

    def forward(self, x):
        x = self.fc(x)
        scores = torch.einsum("bpq,bq->bp", x, x[:, -1])
        scores = torch.nn.functional.softmax(scores, dim=-1)

        x = torch.einsum("bpq,bp->bq", x, scores)
        x = self.fc2(x)

        # return x
        return x.view(-1, n_blocks + 2, n_blocks * 2)


In [37]:
n_dim = 5120

probe = StepProbe(n_dim, 500, n_blocks).to(device)

In [38]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    inputs = [torch.tensor(x["input"], dtype=compute_dtype) for x in batch]
    inputs = pad_sequence(inputs, batch_first=True, padding_value=0, padding_side="left")
    labels = np.stack([x["labels"] for x in batch])
    labels = torch.tensor(labels, dtype=torch.int64)
    return {
        "input": inputs.to(device),
        "labels": labels.to(device)
    }

In [39]:
import torch
import numpy as np
from torch.optim import Adam
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from sklearn.metrics import f1_score

def train_probe(probe, train_dataset, test_dataset, patience=30):
    optimizer = Adam(probe.parameters(), lr=1e-3)
    criterion = CrossEntropyLoss()
    train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, collate_fn=collate_fn)

    n_epochs = 500
    best_f1 = float('-inf')
    early_stop_counter = 0
    
    for epoch in range(n_epochs):
        probe.train()
        total_loss = 0
        n_samples = 0

        for batch in train_loader:
            optimizer.zero_grad()
            input = batch["input"].to(device).float()
            labels = batch["labels"].to(device)
            
            output = probe(input)

            # print(output.shape, labels.shape, input.shape)

            loss = criterion(output, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * len(batch["input"])
            n_samples += len(batch["input"])

        avg_train_loss = total_loss / n_samples
        
        # Evaluation
        probe.eval()
        with torch.no_grad():
            # block_wise_hits = np.zeros((n_blocks * 2), dtype=np.int64)
            block_wise_hits = 0
            total = 0  
            val_loss = 0
            all_preds = []
            all_labels = []
            
            for batch in test_loader:
                input = batch["input"].to(device).float()
                labels = batch["labels"].to(device)
                
                output = probe(input)

                loss = criterion(output, labels)
                val_loss += loss.item() * len(batch["input"])

                preds = output.argmax(dim=1)  # Assuming classification task
                hits = (preds == labels)
                
                block_wise_hits += hits.sum(dim=0).cpu().numpy()
                total += len(labels)
                
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
            
            block_wise_hits = block_wise_hits / total
            
            all_preds = np.concatenate(all_preds)
            all_labels = np.concatenate(all_labels)
            
            # Compute F1 score block-wise
            # block_wise_f1 = np.zeros(n_blocks * 2)
            # for i in range(n_blocks * 2):
            #     block_wise_f1[i] = f1_score(all_labels[:, i], all_preds[:, i], average='macro')
            
            # avg_f1 = block_wise_f1.mean()
            avg_f1 = f1_score(all_labels, all_preds, average='macro')

            val_loss /= total
            
            print(f"Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Hits: {block_wise_hits.mean():.4f}, F1: {avg_f1:.4f}, Val Loss: {val_loss:.4f}")
        
            # Early Stopping Check
            if avg_f1 > best_f1:
                best_f1 = avg_f1
                early_stop_counter = 0
            else:
                early_stop_counter += 1
            
            if early_stop_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break
    
    return block_wise_hits, best_f1


In [ ]:
train_probe(probe, train_dataset, test_dataset)

Epoch 0, Train Loss: 0.6050, Hits: 0.7348, F1: 0.6923, Val Loss: 0.5279
Epoch 1, Train Loss: 0.4301, Hits: 0.7790, F1: 0.7216, Val Loss: 0.4991
Epoch 2, Train Loss: 0.3851, Hits: 0.8065, F1: 0.7695, Val Loss: 0.4357
Epoch 3, Train Loss: 0.3439, Hits: 0.8076, F1: 0.7674, Val Loss: 0.4332
Epoch 4, Train Loss: 0.3250, Hits: 0.8292, F1: 0.7909, Val Loss: 0.4126
Epoch 5, Train Loss: 0.3118, Hits: 0.8280, F1: 0.7970, Val Loss: 0.4007
Epoch 6, Train Loss: 0.2976, Hits: 0.8327, F1: 0.8067, Val Loss: 0.3985
Epoch 7, Train Loss: 0.2870, Hits: 0.8303, F1: 0.7984, Val Loss: 0.4011
Epoch 8, Train Loss: 0.2816, Hits: 0.8292, F1: 0.7999, Val Loss: 0.3932
Epoch 9, Train Loss: 0.2733, Hits: 0.8387, F1: 0.8064, Val Loss: 0.3904
Epoch 10, Train Loss: 0.2667, Hits: 0.8315, F1: 0.7968, Val Loss: 0.3937
Epoch 11, Train Loss: 0.2621, Hits: 0.8363, F1: 0.8026, Val Loss: 0.3945
Epoch 12, Train Loss: 0.2550, Hits: 0.8471, F1: 0.8199, Val Loss: 0.3763
Epoch 13, Train Loss: 0.2529, Hits: 0.8411, F1: 0.8143, Val L

KeyboardInterrupt: 

INFO 02-26 18:29:22 multiproc_worker_utils.py:128] Killing local vLLM worker processes


In [32]:
# print all lines, where the probe is correct
idx = 1367

for ii, item in enumerate(test_dataset.items):
    top_positions = item["top_positions"]
    bottom_positions = item["bottom_positions"]

    top_representations = last_hidden_states[item["idx"]][top_positions].mean(axis=0)
    bottom_representations = last_hidden_states[item["idx"]][bottom_positions].mean(axis=0)

    # hidden_states = self.hidden_states[item["idx"]][post_pos-10:post_pos + 1]
    hidden_states = top_representations - bottom_representations
    above, below = item["above"], item["below"]
    # print(above, below)
    with torch.no_grad():
        input = torch.tensor(hidden_states, dtype=compute_dtype).to(device)
    preds = probe(input.unsqueeze(0)).argmax(dim=1).squeeze().cpu().numpy()

    label = state_to_label((above, below, None))

    if preds.item() == label:
        blocks_label = labels_dict[item["idx"]][item["line_n"]]
        print(blocks_label["line_n"], preds.item())
        print(blocks_label["new_text"])
        print(blocks_label["parsed"]["blocks"])
        print()

            


TypeError: state_to_label() missing 2 required positional arguments: 'top_block' and 'bottom_block'